# IRL & Subgoal Inference: Algorithm Comparison

Applies five algorithms to real LLM-generated coin-gridworld trajectories.

| Algorithm | Output | Predicted cell |
|-----------|--------|---------------|
| MCE | Recovered reward heatmap | argmax of learned reward |
| BIRL | Posterior mean reward heatmap | argmax of posterior mean |
| Inverse Planning | Posterior over subgoal cells | argmax of posterior |
| BNIRL | Gibbs sample histogram | mode of samples |
| Surprise | Log-likelihood surface | argmax of ll_grid |

**Metric:** Manhattan distance from predicted cell to ground-truth `coin_pos`.

**Conditions:** one-shot (each trajectory individually) and aggregated (all 10 together).

In [ ]:
import sys, os, json, io, warnings, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from scipy.special import logsumexp
from tqdm.auto import tqdm

# ── Root paths ──────────────────────────────────────────────────────────────
NB_DIR  = Path(os.path.abspath(''))          # jupyter_demos/analysis/
ROOT    = NB_DIR.parent.parent               # Kiera_code/
DATA_DIR = ROOT / "data_20260525/manhattan_constraint/multiple_trajectories"
PLOT_DIR = DATA_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)

# Add Kiera_code to path for mdp, birl, utils, etc.
sys.path.insert(0, str(ROOT))
# Add inverse_planning dir for bayesian_subgoal_inference_demo
sys.path.insert(0, str(ROOT / "jupyter_demos" / "inverse_planning"))
# Add surprise dir for surprise_model
sys.path.insert(0, str(ROOT / "jupyter_demos" / "surprise"))

from mdp_old import GridMdpOld
from birl import PolicyWalk, uniform_prior
from mdp import policy_iteration, value_function, q_value
from utils import WEST, SOUTH, EAST, NORTH
import contextlib

warnings.filterwarnings("ignore")
print("Imports OK. ROOT =", ROOT)

In [ ]:
# ── Data constants ──────────────────────────────────────────────────────────
PREFIX   = "together_ai_openai_gpt-oss-20b_size7_comp0.2"
N_GRIDS  = 5
N_TRAJS  = 10
ALGO_NAMES = ["MCE", "BIRL", "Inv. Planning", "BNIRL", "Surprise"]

# ── Build-path helper (from plot_coin_trajectories.py) ────────────────────
ACTION_DELTA = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}

def _find_symbol(grid_state_rows, symbol):
    for row_str in grid_state_rows[1:]:
        parts = row_str.split()
        if len(parts) < 2:
            continue
        try:
            row_idx = int(parts[0])
        except ValueError:
            continue
        for col_idx, cell in enumerate(parts[1:]):
            if cell == symbol:
                return (col_idx, row_idx)
    return None

def build_path(steps):
    """Extract (col, row_from_top) path from JSON steps."""
    path = []
    for step in steps:
        pos = _find_symbol(step.get("grid_state", []), "A")
        if pos is None:
            break
        path.append(pos)
    if steps and path:
        last_action = steps[-1].get("agent_action", "").upper()
        dx, dy = ACTION_DELTA.get(last_action, (0, 0))
        path.append((path[-1][0] + dx, path[-1][1] + dy))
    return path

def load_grid_data(grid_id):
    """Return (layout_dict, list_of_10_paths) for grid_id."""
    lf = DATA_DIR / f"{PREFIX}_grid{grid_id}_coin_layout.json"
    with open(lf) as f:
        layout = json.load(f)
    paths = []
    for i in range(N_TRAJS):
        tf = DATA_DIR / f"{PREFIX}_grid{grid_id}_coin_low_traj{i}.json"
        with open(tf) as f:
            data = json.load(f)
        paths.append(build_path(data.get("steps", [])))
    return layout, paths

# Quick smoke-test
layout0, paths0 = load_grid_data(0)
print(f"Grid 0: {len(paths0)} trajectories, coin@{layout0['coin_pos']}, goal@{layout0['goal_pos']}")
print(f"Traj 0 path length: {len(paths0[0])} steps")

In [ ]:
# ── Coordinate helpers ───────────────────────────────────────────────────────
# JSON: (col, row_from_top)  — row 0 = top of display image
# MDP:  (x,  y_mdp)          — y=0 = bottom; GridMdpOld reverses grid internally

def to_mdp(col, row_top, n_rows):
    return (col, n_rows - 1 - row_top)

def to_json(col, y_mdp, n_rows):
    return (col, n_rows - 1 - y_mdp)

def build_gridmdp_old(layout):
    """GridMdpOld from layout JSON. States are (x, y_mdp) tuples."""
    n_rows = len(layout["grid_layout"])
    goal_col, goal_row_top = layout["goal_pos"]
    goal_y = n_rows - 1 - goal_row_top
    raw = [[None if c == "#" else -1.0 for c in row]
           for row in layout["grid_layout"]]   # row 0 = top; GridMdpOld reverses
    return GridMdpOld(raw, terminal_locs=[(goal_col, goal_y)], gamma=0.99)

def json_path_to_mdp_traj(path, n_rows, terminal_marker=True):
    """[(col,row_top),...] → [(state_mdp, action_mdp),...,(terminal,None)]."""
    if len(path) < 2:
        return []
    traj = []
    for i in range(len(path) - 1):
        col, rt = path[i]
        s = to_mdp(col, rt, n_rows)
        dx   = path[i+1][0] - path[i][0]
        dy_t = path[i+1][1] - path[i][1]   # positive = moving down in display
        dy_m = -dy_t                         # flip: MDP y increases upward
        if (dx, dy_m) != (0, 0):            # skip wall-bumps
            traj.append((s, (dx, dy_m)))
    if terminal_marker and path:
        col, rt = path[-1]
        traj.append((to_mdp(col, rt, n_rows), None))
    return traj

def path_to_surprise_traj(path, n_rows):
    """[(col,row_top),...] → [(pos_mdp, action_str),...] for surprise model."""
    _dir_map = {(0, 1): 'N', (0, -1): 'S', (1, 0): 'E', (-1, 0): 'W'}
    traj = []
    for i in range(len(path) - 1):
        col, rt = path[i]
        pos = to_mdp(col, rt, n_rows)
        dx   = path[i+1][0] - path[i][0]
        dy_t = path[i+1][1] - path[i][1]
        dy_m = -dy_t
        act  = _dir_map.get((dx, dy_m))
        if act is not None:
            traj.append((pos, act))
    return traj

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

print("Coord helpers defined.")

In [ ]:
# ── MCE — Tabular Maximum Causal Entropy IRL ────────────────────────────────
ACTIONS_CARD = [WEST, SOUTH, EAST, NORTH]   # cardinal only; no STAY

def run_mce(paths, layout, n_iter=40, lr=0.05, beta=5.0, gamma=0.95):
    """Tabular MCE IRL from trajectory paths (JSON coords).

    Returns (score_grid_display[row_top, col], predicted_json_coords).
    """
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    goal_j = tuple(layout["goal_pos"])        # (col, row_top) JSON
    goal_m = to_mdp(*goal_j, n_rows)          # (col, y_mdp)

    mdp = build_gridmdp_old(layout)
    theta = {s: 0.0 for s in mdp.states}

    # Demo occupancy: discounted state visitation over all paths
    demo_om = {s: 0.0 for s in mdp.states}
    n = len(paths)
    for path in paths:
        for t, (col, rt) in enumerate(path):
            s = to_mdp(col, rt, n_rows)
            if s in demo_om:
                demo_om[s] += (gamma ** t) / n

    # Initial state distribution (uniform over observed starts)
    init_dist = {s: 0.0 for s in mdp.states}
    for path in paths:
        if path:
            s0 = to_mdp(*path[0], n_rows)
            if s0 in init_dist:
                init_dist[s0] += 1.0 / n

    horizon = min(n_rows * n_cols, 60)

    def soft_vi(th):
        V = {s: 0.0 for s in mdp.states}
        for _ in range(300):
            V_new = {}
            for s in mdp.states:
                if s == goal_m:
                    V_new[s] = 0.0
                    continue
                r  = th[s]
                qs = np.array([r + gamma * sum(p * V.get(sp, 0.0)
                               for p, sp in mdp.transitions[s][a])
                               for a in ACTIONS_CARD])
                V_new[s] = logsumexp(beta * qs) / beta
            delta = max(abs(V_new[s] - V.get(s, 0.0)) for s in mdp.states)
            V = V_new
            if delta < 1e-7:
                break
        Q = {}
        for s in mdp.states:
            for a in ACTIONS_CARD:
                if s == goal_m:
                    Q[s, a] = 0.0
                else:
                    Q[s, a] = th[s] + gamma * sum(p * V.get(sp, 0.0)
                                                   for p, sp in mdp.transitions[s][a])
        return V, Q

    def forward_om(V, Q):
        D = {s: float(init_dist.get(s, 0.0)) for s in mdp.states}
        om = {s: 0.0 for s in mdp.states}
        g_t = 1.0
        for _ in range(horizon):
            D_new = {s: 0.0 for s in mdp.states}
            for s in mdp.states:
                d = D[s]
                if d < 1e-15:
                    continue
                om[s] += g_t * d
                if s == goal_m:
                    D_new[s] += d
                    continue
                qs = np.array([Q[s, a] for a in ACTIONS_CARD])
                log_z = logsumexp(beta * qs)
                for ai, a in enumerate(ACTIONS_CARD):
                    pi_sa = np.exp(beta * qs[ai] - log_z)
                    for p, sp in mdp.transitions[s][a]:
                        D_new[sp] = D_new.get(sp, 0.0) + d * pi_sa * p
            D = D_new
            g_t *= gamma
        return om

    for _ in range(n_iter):
        V, Q = soft_vi(theta)
        model_om = forward_om(V, Q)
        for s in mdp.states:
            if s == goal_m:
                continue
            theta[s] += lr * (demo_om.get(s, 0.0) - model_om.get(s, 0.0))

    # Build display-coord score grid [row_top, col]
    score = np.full((n_rows, n_cols), np.nan)
    for (col, y), r in theta.items():
        score[n_rows - 1 - y, col] = r

    # Predicted = argmax excluding goal
    excl = score.copy()
    excl[goal_j[1], goal_j[0]] = np.nan
    idx = int(np.nanargmax(excl))
    r_p, c_p = divmod(idx, n_cols)
    return score, (c_p, r_p)

print("MCE wrapper defined.")

In [ ]:
# ── BIRL — Bayesian IRL via PolicyWalk MCMC ─────────────────────────────────

def run_birl(paths, layout, n_iter=60, burn_in=20,
             r_min=-3.0, r_max=1.0, step_size=0.5, alpha=5.0):
    """PolicyWalk MCMC on GridMdpOld. Returns (score_grid, predicted_json)."""
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    goal_j = tuple(layout["goal_pos"])
    goal_m = to_mdp(*goal_j, n_rows)

    mdp_true = build_gridmdp_old(layout)

    # Trajectories: (state_mdp, action_mdp) pairs WITHOUT terminal marker
    trajs_mdp = [json_path_to_mdp_traj(p, n_rows, terminal_marker=False)
                 for p in paths if len(p) >= 2]
    trajs_mdp = [t for t in trajs_mdp if t]  # drop empties

    if not trajs_mdp:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    pw = PolicyWalk(
        mdp_true=mdp_true, trajectories=trajs_mdp,
        r_min=r_min, r_max=r_max, step_size=step_size,
        alpha=alpha, prior=uniform_prior, iterations=n_iter,
    )

    def log_post(mdp, pi):
        V = value_function(mdp, pi)
        ll = sum(q_value(mdp, s, a, V=V)
                 for s, a in pw.trajectories if a is not None)
        return pw.alpha * ll + uniform_prior(mdp, pw.r_min, pw.r_max)

    # Initialise
    mdp_cur = deepcopy(mdp_true)
    mdp_cur.rewards = pw.random_rewards()
    pi_cur = policy_iteration(mdp_cur)
    lp_cur = log_post(mdp_cur, pi_cur)

    rewards_trace = []
    for i in range(n_iter):
        pw.current_i = i
        mdp_prop = deepcopy(mdp_cur)
        mdp_prop.rewards = pw.get_rewards_neighbour(mdp_cur.rewards)
        pi_prop = policy_iteration(mdp_prop, pi_cur)
        lp_prop = log_post(mdp_prop, pi_prop)

        ratio = np.exp(np.clip(lp_prop - lp_cur, -500, 500))
        if np.random.random() < min(1.0, ratio):
            mdp_cur, pi_cur, lp_cur = mdp_prop, pi_prop, lp_prop

        rewards_trace.append(deepcopy(mdp_cur.rewards))

    # Posterior mean reward
    mean_r = {s: float(np.mean([r[s] for r in rewards_trace[burn_in:]]))
              for s in mdp_true.states}

    score = np.full((n_rows, n_cols), np.nan)
    for (col, y), r in mean_r.items():
        score[n_rows - 1 - y, col] = r

    excl = score.copy()
    excl[goal_j[1], goal_j[0]] = np.nan
    idx  = int(np.nanargmax(excl))
    r_p, c_p = divmod(idx, n_cols)
    return score, (c_p, r_p)

print("BIRL wrapper defined.")

In [ ]:
# ── Inverse Planning — Bayesian subgoal inference ────────────────────────────
# Import from bayesian_subgoal_inference_demo.py
from bayesian_subgoal_inference_demo import (
    soft_value_iteration, compute_posterior, candidate_set,
)

def run_inv_planning(paths, layout, beta=2.0):
    """Bayesian subgoal inference via soft value iteration.

    Returns (score_grid_display[row_top,col], predicted_json_coords).
    """
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    goal_j = tuple(layout["goal_pos"])
    goal_m = to_mdp(*goal_j, n_rows)

    mdp = build_gridmdp_old(layout)

    # Convert paths to MDP trajectories (with terminal marker)
    trajs_mdp = [json_path_to_mdp_traj(p, n_rows, terminal_marker=True)
                 for p in paths if len(p) >= 2]
    trajs_mdp = [t for t in trajs_mdp if t]

    if not trajs_mdp:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    starts = [t[0][0] for t in trajs_mdp]   # first state of each traj
    cands  = candidate_set(trajs_mdp, starts, goal_m)

    if not cands:
        # Fall back: union of all visited non-start, non-terminal states
        excluded = set(starts) | {goal_m}
        cands = frozenset(
            s for t in trajs_mdp for s, _ in t if s not in excluded
        )

    if not cands:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    # Suppress the print inside compute_posterior
    with contextlib.redirect_stdout(io.StringIO()):
        posterior = compute_posterior(trajs_mdp, mdp, cands, goal_m, beta)

    # Build display score grid
    score = np.full((n_rows, n_cols), np.nan)
    for (col, y), prob in posterior.items():
        score[n_rows - 1 - y, col] = prob

    best_m = max(posterior, key=posterior.get)
    return score, to_json(*best_m, n_rows)

print("Inverse Planning wrapper defined.")

In [ ]:
# ── BNIRL — Fixed-K=2 Gibbs Sampler ─────────────────────────────────────────
# Adapted from jupyter_demos/BNIRL/append_sections.py
# Key change: GRID global → explicit (grid_rows, grid_cols) parameters

def _bnirl_step_lik(pos, next_pos, goal, alpha, grid_rows, grid_cols):
    px, py = int(pos[0]),      int(pos[1])
    nx, ny = int(next_pos[0]), int(next_pos[1])
    gx, gy = int(goal[0]),     int(goal[1])
    nbrs = [(px+d[0], py+d[1])
            for d in [(0,1),(0,-1),(1,0),(-1,0)]
            if 0 <= px+d[0] < grid_cols and 0 <= py+d[1] < grid_rows]
    if not nbrs:
        return 1.0
    scores = np.array([alpha * (-abs(cx-gx) - abs(cy-gy))
                       for cx, cy in nbrs], dtype=float)
    scores -= scores.max()
    w = np.exp(scores)
    w /= w.sum()
    for i, (cx, cy) in enumerate(nbrs):
        if cx == nx and cy == ny:
            return float(w[i])
    return 1e-10

def _bnirl_sample_assignments(steps, subgoal, terminal, alpha, rng, gr, gc):
    n  = len(steps) - 1
    sg = tuple(int(x) for x in subgoal)
    tg = tuple(int(x) for x in terminal)
    assignments = np.zeros(n, dtype=int)
    for i in range(n):
        ll_sg = _bnirl_step_lik(steps[i], steps[i+1], sg, alpha, gr, gc)
        ll_tg = _bnirl_step_lik(steps[i], steps[i+1], tg, alpha, gr, gc)
        total = ll_sg + ll_tg
        p_sg  = ll_sg / total if total > 1e-15 else 0.5
        assignments[i] = rng.choice(2, p=[p_sg, 1.0 - p_sg])
    return assignments

def _bnirl_sample_subgoal(steps, assignments, rng, gr, gc):
    mask = (assignments == 0)
    if mask.sum() == 0:
        return (int(rng.integers(0, gc)), int(rng.integers(0, gr)))
    ends = np.array([[int(steps[i+1][0]), int(steps[i+1][1])]
                     for i in range(len(assignments)) if assignments[i] == 0])
    mx = int(np.clip(round(float(np.median(ends[:, 0]))), 0, gc - 1))
    my = int(np.clip(round(float(np.median(ends[:, 1]))), 0, gr - 1))
    return (mx, my)

def _run_gibbs(steps, terminal, n_iter, alpha, rng, gr, gc):
    init  = int(rng.integers(0, len(steps)))
    sg    = tuple(int(x) for x in steps[init])
    tg    = tuple(int(x) for x in terminal)
    hist  = np.zeros((n_iter, 2), dtype=int)
    for t in range(n_iter):
        asgn  = _bnirl_sample_assignments(steps, sg, tg, alpha, rng, gr, gc)
        sg    = _bnirl_sample_subgoal(steps, asgn, rng, gr, gc)
        hist[t] = sg
    return hist

def run_bnirl(paths, layout, n_iter=500, burn_in=100, alpha=3.0, seed=42):
    """Fixed-K=2 Gibbs sampler for subgoal inference.

    Returns (score_grid_display[row_top,col], predicted_json_coords).
    """
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    goal_j = tuple(layout["goal_pos"])
    goal_m = to_mdp(*goal_j, n_rows)

    # Position sequence in MDP coords (all trajectories concatenated)
    steps = []
    for path in paths:
        for col, rt in path:
            steps.append(to_mdp(col, rt, n_rows))

    if len(steps) < 2:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    rng  = np.random.default_rng(seed)
    hist = _run_gibbs(steps, goal_m, n_iter, alpha, rng, n_rows, n_cols)

    # 2-D histogram over post-burn-in samples; indexed [y_mdp, col]
    heat = np.zeros((n_rows, n_cols), dtype=float)
    for x, y in hist[burn_in:]:
        if 0 <= x < n_cols and 0 <= y < n_rows:
            heat[y, x] += 1

    # Display grid: flip y axis (y_mdp=0 → row_top=n_rows-1)
    score = heat[::-1].copy().astype(float)
    # Mask walls
    grid = layout["grid_layout"]
    for rt in range(n_rows):
        for col in range(n_cols):
            if grid[rt][col] == "#":
                score[rt, col] = np.nan

    if np.nanmax(score) == 0:
        return score, goal_j

    excl = score.copy()
    excl[goal_j[1], goal_j[0]] = np.nan
    idx  = int(np.nanargmax(excl))
    r_p, c_p = divmod(idx, n_cols)
    return score, (c_p, r_p)

print("BNIRL wrapper defined.")

In [ ]:
# ── Surprise Model — Shannon Surprise goal inference ─────────────────────────
from surprise_model import infer_subgoal, DEFAULT_PARAMS

def run_surprise(paths, layout, model='surprise', params=DEFAULT_PARAMS):
    """Surprise model subgoal inference.

    Returns (score_grid_display[row_top,col], predicted_json_coords).
    ll_grid from infer_subgoal is indexed [y_mdp][x]; flip y for display.
    """
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    goal_j = tuple(layout["goal_pos"])
    goal_m = to_mdp(*goal_j, n_rows)
    grid   = layout["grid_layout"]

    # Surprise model expects square grid of size n; use the larger dimension
    grid_size = max(n_rows, n_cols)

    # Convert paths to surprise trajectory format [(pos_mdp, action_str),...]
    trajs_s = [path_to_surprise_traj(p, n_rows) for p in paths]
    trajs_s = [t for t in trajs_s if t]

    if not trajs_s:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    # Candidates: non-wall, non-terminal, non-start cells in MDP coords
    start_set = {to_mdp(*p[0], n_rows) for p in paths if p}
    candidates = [
        to_mdp(col, rt, n_rows)
        for rt in range(n_rows) for col in range(n_cols)
        if grid[rt][col] != "#"
        and to_mdp(col, rt, n_rows) != goal_m
        and to_mdp(col, rt, n_rows) not in start_set
    ]

    if not candidates:
        score = np.full((n_rows, n_cols), np.nan)
        return score, goal_j

    with contextlib.redirect_stdout(io.StringIO()):
        ll_grid, best_m = infer_subgoal(
            trajs_s, goal_m, candidates, model=model,
            params=params, grid_size=grid_size,
        )

    # ll_grid indexed [y_mdp][x]; flip → display [row_top][col]
    score = np.full((n_rows, n_cols), np.nan)
    for y_m in range(n_rows):
        for col in range(n_cols):
            v = ll_grid[y_m][col] if col < ll_grid.shape[1] and y_m < ll_grid.shape[0] else np.nan
            score[n_rows - 1 - y_m, col] = v

    return score, to_json(*best_m, n_rows)

print("Surprise wrapper defined.")

In [ ]:
# ── Plotting helper ──────────────────────────────────────────────────────────

def plot_result(ax, score_grid, layout, predicted_json, title, mode='reward'):
    """Draw one algorithm result panel onto ax.

    mode='reward'  → diverging RdYlGn colormap  (MCE, BIRL)
    mode='subgoal' → sequential YlOrRd colormap  (InvPlan, BNIRL, Surprise)
    """
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])
    grid   = layout["grid_layout"]
    coin_j = tuple(layout["coin_pos"])    # ground truth
    goal_j = tuple(layout["goal_pos"])
    start_j = tuple(layout["agent_start_pos"])

    # Mask walls
    wall_mask = np.array([[grid[r][c] == "#"
                           for c in range(n_cols)]
                          for r in range(n_rows)])
    masked = np.ma.masked_where(wall_mask, score_grid)

    if mode == 'reward':
        finite = score_grid[~np.isnan(score_grid) & ~wall_mask]
        if len(finite) == 0 or finite.min() == finite.max():
            norm = None
        else:
            vmin, vmax = float(finite.min()), float(finite.max())
            norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax) if vmin < 0 < vmax else None
        cmap = plt.cm.RdYlGn.copy()
    else:
        norm = None
        cmap = plt.cm.YlOrRd.copy()

    cmap.set_bad("#4a4a4a")
    im = ax.imshow(masked, origin='upper', cmap=cmap, norm=norm,
                   interpolation='nearest', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # Grid lines
    for r in range(n_rows + 1):
        ax.axhline(r - 0.5, color='#888', lw=0.3, zorder=1)
    for c in range(n_cols + 1):
        ax.axvline(c - 0.5, color='#888', lw=0.3, zorder=1)

    # Labels
    ax.text(goal_j[0],  goal_j[1],  'G', ha='center', va='center',
            fontsize=7, fontweight='bold', color='black', zorder=5)
    ax.text(start_j[0], start_j[1], 'S', ha='center', va='center',
            fontsize=7, fontweight='bold', color='black', zorder=5)
    ax.text(coin_j[0],  coin_j[1],  'C', ha='center', va='center',
            fontsize=7, fontweight='bold', color='black', zorder=5)

    # Ground-truth coin: green circle
    ax.add_patch(mpatches.Circle(coin_j, 0.35, color='limegreen',
                                 fill=False, lw=1.5, zorder=6))
    # Predicted: red X
    ax.plot(predicted_json[0], predicted_json[1], 'x',
            color='crimson', ms=8, mew=2, zorder=7)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=6, pad=2)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)

print("plot_result defined.")

In [ ]:
# ── Main runner: all algorithms × one grid ───────────────────────────────────

ALGO_FUNCS = [run_mce, run_birl, run_inv_planning, run_bnirl, run_surprise]
ALGO_MODES = ['reward', 'reward', 'subgoal', 'subgoal', 'subgoal']

def run_grid(grid_id, verbose=True):
    """Run all 5 algorithms (one-shot + aggregated) on grid_id.

    Saves grid{N}_oneshot.png and grid{N}_aggregated.png to PLOT_DIR.
    Returns distances dict:
      {'oneshot': {algo: [dist_traj0..9]}, 'aggregated': {algo: dist}}
    """
    layout, paths = load_grid_data(grid_id)
    coin_j = tuple(layout["coin_pos"])
    n_rows = len(layout["grid_layout"])
    n_cols = len(layout["grid_layout"][0])

    dists = {'oneshot': {a: [] for a in ALGO_NAMES},
             'aggregated': {a: None for a in ALGO_NAMES}}

    # ── ONE-SHOT: 10 trajectories × 5 algorithms ─────────────────────────
    n_rows_fig = N_TRAJS         # 10 rows
    n_cols_fig = len(ALGO_NAMES) # 5 cols
    panel_w, panel_h = max(2.0, n_cols * 0.35), max(2.0, n_rows * 0.35)
    fig_os, axes_os = plt.subplots(
        n_rows_fig, n_cols_fig,
        figsize=(n_cols_fig * panel_w + 0.3, n_rows_fig * panel_h + 0.8),
    )

    for ti, path in enumerate(paths):
        for ai, (algo_fn, algo_name, mode) in enumerate(
                zip(ALGO_FUNCS, ALGO_NAMES, ALGO_MODES)):
            ax = axes_os[ti, ai]
            try:
                score, pred = algo_fn([path], layout)
                dist = manhattan(pred, coin_j)
            except Exception as e:
                score = np.full((n_rows, n_cols), np.nan)
                pred  = tuple(layout["goal_pos"])
                dist  = manhattan(pred, coin_j)
                if verbose:
                    print(f"  [grid{grid_id} traj{ti} {algo_name}] ERROR: {e}")
            dists['oneshot'][algo_name].append(dist)
            plot_result(ax, score, layout, pred,
                        f"{algo_name}  t{ti}  d={dist}", mode=mode)

    # Column headers
    for ai, name in enumerate(ALGO_NAMES):
        axes_os[0, ai].set_title(f"{name}", fontsize=7, pad=3, fontweight='bold')

    fig_os.suptitle(f"Grid {grid_id} — One-shot  |  coin={coin_j}",
                    fontsize=8, y=1.005)
    fig_os.tight_layout()
    out = PLOT_DIR / f"grid{grid_id}_oneshot.png"
    fig_os.savefig(out, dpi=120, bbox_inches='tight')
    plt.close(fig_os)
    if verbose:
        print(f"Saved {out.name}")

    # ── AGGREGATED: all 10 trajectories → 5 algorithms ───────────────────
    fig_ag, axes_ag = plt.subplots(
        1, n_cols_fig,
        figsize=(n_cols_fig * panel_w + 0.3, panel_h + 0.8),
    )

    for ai, (algo_fn, algo_name, mode) in enumerate(
            zip(ALGO_FUNCS, ALGO_NAMES, ALGO_MODES)):
        ax = axes_ag[ai]
        try:
            score, pred = algo_fn(paths, layout)
            dist = manhattan(pred, coin_j)
        except Exception as e:
            score = np.full((n_rows, n_cols), np.nan)
            pred  = tuple(layout["goal_pos"])
            dist  = manhattan(pred, coin_j)
            if verbose:
                print(f"  [grid{grid_id} agg {algo_name}] ERROR: {e}")
        dists['aggregated'][algo_name] = dist
        plot_result(ax, score, layout, pred,
                    f"{algo_name}  agg  d={dist}", mode=mode)

    fig_ag.suptitle(f"Grid {grid_id} — Aggregated (all 10 trajs)  |  coin={coin_j}",
                    fontsize=8, y=1.01)
    fig_ag.tight_layout()
    out = PLOT_DIR / f"grid{grid_id}_aggregated.png"
    fig_ag.savefig(out, dpi=120, bbox_inches='tight')
    plt.close(fig_ag)
    if verbose:
        print(f"Saved {out.name}")

    # Print summary table
    if verbose:
        print(f"\n  Grid {grid_id} distances (coin @ {coin_j}):")
        print(f"  {'Algorithm':<16} {'One-shot mean±std':>20}  {'Aggregated':>10}")
        for name in ALGO_NAMES:
            os_d = dists['oneshot'][name]
            agg_d = dists['aggregated'][name]
            print(f"  {name:<16} {np.mean(os_d):>8.2f} ± {np.std(os_d):<8.2f}  {agg_d:>10}")

    return dists

print("run_grid defined. Ready to run.")

## Grid 0–4 Results

Run each grid cell independently. Each cell saves two PNGs to `plots/` and prints a distance table.

> **Runtime note:** MCE ~4s/call · BIRL ~12s/call · others <1s/call → ~3 min per grid cell.

In [ ]:
all_dists = {}   # populated as grids run; used by summary cell below

dists0 = run_grid(0)
all_dists[0] = dists0

In [ ]:
dists1 = run_grid(1)
all_dists[1] = dists1

In [ ]:
dists2 = run_grid(2)
all_dists[2] = dists2

In [ ]:
dists3 = run_grid(3)
all_dists[3] = dists3

In [ ]:
dists4 = run_grid(4)
all_dists[4] = dists4

In [ ]:
# ── Summary distance bar chart ────────────────────────────────────────────────
# Requires all 5 grid cells above to have been run.

if len(all_dists) == N_GRIDS:
    # One-shot: mean across grids of (mean across trajs)
    os_means  = {a: [] for a in ALGO_NAMES}
    os_stds   = {a: [] for a in ALGO_NAMES}
    agg_means = {a: [] for a in ALGO_NAMES}

    for gid in range(N_GRIDS):
        d = all_dists[gid]
        for name in ALGO_NAMES:
            os_vals = d['oneshot'][name]
            os_means[name].append(np.mean(os_vals))
            os_stds[name].append(np.std(os_vals))
            agg_means[name].append(d['aggregated'][name])

    # Average over grids
    mean_os  = {a: np.mean(os_means[a])  for a in ALGO_NAMES}
    mean_agg = {a: np.mean(agg_means[a]) for a in ALGO_NAMES}
    std_os   = {a: np.mean(os_stds[a])   for a in ALGO_NAMES}

    x      = np.arange(len(ALGO_NAMES))
    width  = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    bars1 = ax.bar(x - width/2, [mean_os[a]  for a in ALGO_NAMES], width,
                   label='One-shot (mean ± std)', color='steelblue',
                   yerr=[std_os[a] for a in ALGO_NAMES], capsize=4)
    bars2 = ax.bar(x + width/2, [mean_agg[a] for a in ALGO_NAMES], width,
                   label='Aggregated', color='darkorange')

    ax.set_xticks(x)
    ax.set_xticklabels(ALGO_NAMES)
    ax.set_ylabel("Manhattan distance to coin_pos")
    ax.set_title(f"Algorithm comparison — mean over {N_GRIDS} grids\n"
                 "(lower = better; green circle = coin ground truth)")
    ax.legend()
    ax.axhline(0, color='gray', lw=0.5)
    fig.tight_layout()
    out = PLOT_DIR / "summary_distances.png"
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"\nSaved {out}")

    # Print table
    print(f"\n{'Algorithm':<16} {'OS mean':>8} {'OS std':>8} {'Agg mean':>10}")
    for name in ALGO_NAMES:
        print(f"{name:<16} {mean_os[name]:>8.2f} {std_os[name]:>8.2f} {mean_agg[name]:>10.2f}")
else:
    print(f"Run all {N_GRIDS} grid cells first (have {len(all_dists)} so far).")